# New uncertainty workflow
This notebook mirrors the new-data workflow: select a subset of shoreline shapefiles, pull `CPS` from each shoreline shapefile's attribute table, resolve `Pixel_ER` and `Photoscale` from source metadata, derive `Georef_ER`, and compute `Total_UNCY = sqrt(Ep^2 + Eg^2 + Ed^2)`.

`Photoscale` is not part of the final equation directly; it is used to determine `Georef_ER` for source types that need it.

In [24]:
%load_ext autotime
import geopandas as gpd
import pandas as pd
import numpy as np
import math
import re
from pathlib import Path
from glob import glob
from tqdm.auto import tqdm
import platform
import rasterio
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 130)

if platform.system() == 'Windows':
    prefix = r'Z:/'
else:
    prefix = 'ressci201900060-RNC2-Coastal/'

# EDIT TARGET SHORELINES HERE
# See Part 6 of GETTING_STARTED.md; use the same RUN_OWNER and search criteria in all 3 notebooks.
# The cutoff can be either a single date (e.g. "2024-07-18") or a date range as a
# two-item tuple/list e.g. ("2024-07-18", "2024-08-18") or ["AOI1", "AOI2"]
cutoff_date = ("2024-07-18")
search_roots = [Path(r'Z:\MaxarImagery\HighFreq'), Path(r'Z:\Retrolens')]
search_mode = 'aoi'  # 'date', 'aoi', 'aoi_in_date_range', 'region', or 'region_in_date_range'
target_aoi = ["TeAtatu","Hobsonville","PollenIsland","ShoalBay","SoldiersBay","Ngataringa","FrenchmansBay","ManukapuaIsland","OmokoritoBay","OrongoPoint","Pouto","ShellyBeach","TeHakono_clarksBay","Tinopai"]
target_region = 'Auckland'

# Put your own name here. All three notebooks must use the same value, so your outputs
# stay in your own folder and can never overwrite someone else's run of the same area.
RUN_OWNER = 'catriona'
DATA_DIR = Path('DataUpdatev2') / RUN_OWNER
DATA_DIR.mkdir(parents=True, exist_ok=True)


def _norm(s):
    return ''.join(ch for ch in str(s).lower() if ch.isalnum())


def _as_target_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        values = [str(v).strip() for v in value if str(v).strip()]
    else:
        values = [str(value).strip()]
    return [v for v in values if v]


def _coerce_cutoff_bounds(cutoff):
    if cutoff is None:
        raise ValueError('cutoff_date must not be None')

    if isinstance(cutoff, (tuple, list)):
        if len(cutoff) != 2:
            raise ValueError('cutoff_date range must be a two-item tuple/list')
        start = pd.Timestamp(cutoff[0]).normalize()
        end = pd.Timestamp(cutoff[1]).normalize()
        if start > end:
            start, end = end, start
        return start, end

    single = pd.Timestamp(cutoff).normalize()
    return single, None


def _matches_date(modified, cutoff_start, cutoff_end):
    if cutoff_end is None:
        return modified > cutoff_start
    return (modified >= cutoff_start) and (modified <= cutoff_end)


def _cutoff_label(cutoff_start, cutoff_end):
    if cutoff_end is None:
        return f'modified > {cutoff_start.date()}'
    return f'modified between {cutoff_start.date()} and {cutoff_end.date()}'


valid_modes = {'date', 'aoi', 'aoi_in_date_range', 'region', 'region_in_date_range'}
if search_mode not in valid_modes:
    raise ValueError(f'search_mode must be one of {sorted(valid_modes)}')

if search_mode in {'aoi', 'aoi_in_date_range'} and not _as_target_list(target_aoi):
    raise ValueError('target_aoi must be set when using AOI-based modes')
if search_mode in {'region', 'region_in_date_range'} and not _as_target_list(target_region):
    raise ValueError('target_region must be set when using region-based modes')

cutoff_start, cutoff_end = _coerce_cutoff_bounds(cutoff_date)
target_aoi_norms = {_norm(v) for v in _as_target_list(target_aoi)}
target_region_norms = {_norm(v) for v in _as_target_list(target_region)}
records = []

for root in search_roots:
    if not root.exists():
        continue
    for shp in root.glob('**/Shorelines/*.shp'):
        if shp.stem.lower().startswith('[aoierr]'):
            continue
        if len(shp.parts) < 5 or shp.parts[-2].lower() != 'shorelines':
            continue

        region = shp.parts[-4]
        aoi = shp.parts[-3]
        modified = pd.Timestamp(shp.stat().st_mtime, unit='s')
        stem_aoi = shp.stem.rsplit('_', 1)[0]

        matches_aoi = bool(target_aoi_norms & {_norm(aoi), _norm(stem_aoi)})
        matches_region = bool(target_region_norms & {_norm(region)})
        matches_date = _matches_date(modified, cutoff_start, cutoff_end)

        include = False
        if search_mode == 'date':
            include = matches_date
        elif search_mode == 'aoi':
            include = matches_aoi
        elif search_mode == 'aoi_in_date_range':
            include = matches_aoi and matches_date
        elif search_mode == 'region':
            include = matches_region
        elif search_mode == 'region_in_date_range':
            include = matches_region and matches_date

        if include:
            records.append({
                'source_root': str(root),
                'region': region,
                'aoi': aoi,
                'shoreline_path': str(shp),
                'modified': modified,
            })

new_shorelines = pd.DataFrame(records, columns=['source_root', 'region', 'aoi', 'shoreline_path', 'modified'])
if not new_shorelines.empty:
    new_shorelines = new_shorelines.sort_values(['region', 'aoi', 'modified']).reset_index(drop=True)

mode_label = {
    'date': f'Date range ({_cutoff_label(cutoff_start, cutoff_end)})',
    'aoi': f'AOI only ({target_aoi})',
    'aoi_in_date_range': f'AOI in date range ({target_aoi}; {_cutoff_label(cutoff_start, cutoff_end)})',
    'region': f'Region only ({target_region})',
    'region_in_date_range': f'Region in date range ({target_region}; {_cutoff_label(cutoff_start, cutoff_end)})',
}[search_mode]

print(f'Mode: {mode_label} | Matches: {len(new_shorelines)}')
new_shorelines

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
Mode: AOI only (['TeAtatu', 'Hobsonville', 'PollenIsland', 'ShoalBay', 'SoldiersBay', 'Ngataringa', 'FrenchmansBay', 'ManukapuaIsland', 'OmokoritoBay', 'OrongoPoint', 'Pouto', 'ShellyBeach', 'TeHakono_clarksBay', 'Tinopai']) | Matches: 67


,source_root,region,aoi,shoreline_path,modified
0,Z:\Retrolens,Auckland,Hobsonville,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,2026-08-19 03:42:38.789324760
1,Z:\Retrolens,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_12APR1973.shp,2022-01-16 02:09:57.307988882
2,Z:\Retrolens,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_09MAR2011.shp,2026-08-19 03:42:39.147778034
3,Z:\Retrolens,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_07FEB1982.shp,2026-08-19 03:42:39.618799686
4,Z:\Retrolens,Auckland,ManukapuaIsland,Z:\Retrolens\Auckland\ManukapuaIsland\Shorelines\ManukapuaIsland_16DEC1976.shp,2026-08-19 03:42:40.014433622
...,...,...,...,...,...
62,Z:\Retrolens,Northland,Tinopai,Z:\Retrolens\Northland\Tinopai\Shorelines\Tinopai_18MAR2024.shp,2026-08-19 03:43:04.369038343
63,Z:\Retrolens,Northland,Tinopai,Z:\Retrolens\Northland\Tinopai\Shorelines\Tinopai_17JUN1966.shp,2026-08-19 03:43:04.759257793
64,Z:\Retrolens,Northland,Tinopai,Z:\Retrolens\Northland\Tinopai\Shorelines\Tinopai_24OCT1953.shp,2026-08-19 03:43:05.165162086
65,Z:\Retrolens,Northland,Tinopai,Z:\Retrolens\Northland\Tinopai\Shorelines\Tinopai_16DEC1976.shp,2026-08-19 03:43:05.605516195


time: 2min 27s (started: 2026-08-19 16:21:23 +12:00)


In [25]:
import ast
from pathlib import Path
from typing import Optional
from difflib import SequenceMatcher

try:
    from rapidfuzz import process as rapidfuzz_process
except ImportError:
    rapidfuzz_process = None

CPS_error_lookup = {1: 0.43, 2: 0.73, 3: 0.97, 4: 2.07, 5: 8.59}


def extract_best_match(query, choices):
    choices = list(choices)
    if len(choices) == 0:
        return None, 0, None
    if rapidfuzz_process is not None:
        return rapidfuzz_process.extractOne(query=query, choices=choices)
    query_text = str(query).lower()
    best_choice = None
    best_score = -1.0
    for choice in choices:
        score = SequenceMatcher(None, query_text, str(choice).lower()).ratio()
        if score > best_score:
            best_choice = choice
            best_score = score
    return best_choice, int(best_score * 100), None


def norm_text(value: str) -> str:
    return ''.join(ch for ch in str(value).lower() if ch.isalnum())


def to_source_relpath(filename: str) -> str:
    text = str(filename).replace('\\', '/')
    for marker in ('MaxarImagery/HighFreq/', 'Retrolens/', 'LDS/', 'Archive/Gabrielle/', 'skyvuw/'):
        if marker in text:
            return text[text.index(marker):]
    if text.startswith('Z:/'):
        return text[3:]
    return text.lstrip('/')


SOURCE_ALIASES = {
    'LDS': 'LDS', 'LZ': 'LDS', 'LINZ': 'LDS',
    'RL': 'RL', 'RLN': 'RL', 'RLS': 'RL', 'RS': 'RL', 'RETROLENS': 'RL',
    'MAX': 'MAX', 'MAXAR': 'MAX',
}


def source_from_attributes(shapefile: Optional[gpd.GeoDataFrame]) -> Optional[str]:
    # Only a real Source column counts; Interger/Digitiser hold digitiser initials.
    if shapefile is None or 'Source' not in shapefile.columns:
        return None
    values = shapefile['Source'].dropna().astype(str).str.strip().str.upper()
    values = values[values.ne('') & values.ne('NAN')]
    if values.empty:
        return None
    code = values.value_counts().idxmax()
    return SOURCE_ALIASES.get(code, code)


def get_source(filename: str, shapefile: Optional[gpd.GeoDataFrame] = None) -> str:
    # Prefer the digitiser's own Source attribute, then fall back to the folder the file sits in.
    attr_source = source_from_attributes(shapefile)
    if attr_source:
        return attr_source

    rel = to_source_relpath(filename)
    if rel.startswith('Retrolens/') or rel.startswith('RL/') or 'Retrolens/' in rel:
        return 'RL'
    if rel.startswith('MaxarImagery/HighFreq/') or 'MaxarImagery/HighFreq/' in rel:
        return 'MAX'
    if rel.startswith('LDS/') or '/LDS/' in rel:
        return 'LDS'
    return 'Unknown'


def parse_resolution(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (tuple, list, np.ndarray)):
        return float(value[0]) if len(value) else pd.NA
    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (tuple, list)) and len(parsed):
            return float(parsed[0])
    except Exception:
        pass
    if ',' in text:
        return float(text.replace('(', '').replace(')', '').split(',')[0].strip())
    try:
        return float(text)
    except Exception:
        return pd.NA


def get_georef_er(scale, taranaki=False):
    # Photoscale thresholds use horizontal-accuracy georef values provided by the spec.
    if pd.isna(scale) or not scale:
        return pd.NA
    scale = float(scale)
    if scale < 20000:
        return 2.09
    elif scale < 30000:
        return 2.43
    return 2.90


def normalize_month_typos(token: str) -> str:
    # Known month typos/variants in historical filenames.
    replacements = {
        'APRL': 'APR',
        'SEPT': 'SEP',
        'JUNE': 'JUN',
        'JULY': 'JUL',
    }
    out = str(token).upper()
    for bad, good in replacements.items():
        out = out.replace(bad, good)
    return out


def parse_date_from_stem(stem: str):
    s = str(stem)

    m = re.search(r'(\d{1,2}[A-Za-z]{3,4}\d{4})', s)
    if m:
        token = normalize_month_typos(m.group(1))
        for fmt in ('%d%b%Y', '%d%B%Y'):
            try:
                return pd.to_datetime(token, format=fmt).date()
            except Exception:
                pass

    m = re.search(r'(\d{4}-\d{2}-\d{2})', s)
    if m:
        try:
            return pd.to_datetime(m.group(1), format='%Y-%m-%d').date()
        except Exception:
            pass

    m = re.search(r'(\d{8})', s)
    if m:
        try:
            return pd.to_datetime(m.group(1), format='%Y%m%d').date()
        except Exception:
            pass

    return None


def parse_dsas_date_from_filename(filename: str):
    dt = parse_date_from_stem(Path(filename).stem)
    if dt is None:
        return None
    return pd.Timestamp(dt).strftime('%d/%m/%Y').lstrip('0')


def get_scale(filename, dsas_date, year):
    path = Path(filename)
    parts = path.parts
    terminator_candidates = [parts.index(name) for name in ('Stack', 'Shorelines') if name in parts]
    if not terminator_candidates:
        raise ValueError(f'Could not find Stack/Shorelines in {filename}')
    terminator = min(terminator_candidates)
    root_dir = Path(*parts[:terminator])
    csv_candidates = list(root_dir.glob('*.csv'))
    if len(csv_candidates) == 0:
        raise ValueError(f'No CSV found for {root_dir}')
    if len(csv_candidates) > 1:
        csv_candidates = [csv_candidates[0]]
    csv_filename = csv_candidates[0]
    try:
        csv = pd.read_csv(csv_filename, encoding='cp1252')
    except UnicodeDecodeError:
        csv = pd.read_excel(csv_filename)
        if 'Date' in csv.columns:
            csv['Date'] = csv['Date'].astype(str)

    matched_date, score, _ = extract_best_match(query=dsas_date, choices=csv.Date.dropna().astype(str).unique())
    if score < 80:
        matched_date, score, _ = extract_best_match(query=year, choices=csv.Date.dropna().astype(str).unique())

    if 'RMSE' in csv.columns:
        filtered = csv[(csv.Date.astype(str) == matched_date) & ~csv.RMSE.isna()]
    else:
        filtered = csv[(csv.Date.astype(str) == matched_date)]

    scales = filtered.Scale.unique() if 'Scale' in filtered.columns else []
    if len(scales) == 0 and 'Scale' in csv.columns:
        filtered = csv[csv.Date.astype(str).str.contains(str(matched_date), na=False)]
        scales = filtered.Scale.unique()
    if len(scales) == 0:
        filtered = csv[csv.Date.astype(str).str.contains(str(year), na=False)]
        scales = filtered.Scale.unique() if 'Scale' in filtered.columns else []
    if len(scales) > 1:
        scales = filtered.Scale.value_counts()
        scales = [scales.index[0]]
    if len(scales) == 0:
        raise ValueError(f"Can't find a scale for {filename}")
    return scales[0]


def infer_stack_dirs(shoreline_path: str):
    p = Path(shoreline_path)
    parts = list(p.parts)
    candidates = []
    if 'Shorelines' in parts:
        i = parts.index('Shorelines')
        aoi_root = Path(*parts[:i])
        candidates.append(aoi_root / 'Stack')
        candidates.append(aoi_root / 'Imagery' / 'Stack')
    else:
        parent = p.parent
        candidates.append(parent / 'Stack')
        candidates.append(parent / 'Imagery' / 'Stack')
    uniq = []
    seen = set()
    for c in candidates:
        key = str(c).lower()
        if key not in seen:
            seen.add(key)
            uniq.append(c)
    return uniq


def pick_stack_raster(shoreline_path: str):
    stem = Path(shoreline_path).stem
    stem_norm = norm_text(stem)
    shoreline_date = parse_date_from_stem(stem)
    year_match = re.search(r'(\d{4})', stem)
    year = year_match.group(1) if year_match else None

    for stack_dir in infer_stack_dirs(shoreline_path):
        if not stack_dir.exists():
            continue

        rasters = []
        for ext in ('*.jp2', '*.JP2', '*.tif', '*.TIF', '*.tiff', '*.TIFF'):
            rasters.extend(stack_dir.glob(ext))
        if len(rasters) == 0:
            continue

        # 1) Strict DSAS-date match first.
        if shoreline_date is not None:
            exact_date = [r for r in rasters if parse_date_from_stem(r.stem) == shoreline_date]
            if len(exact_date) == 1:
                return exact_date[0], 'stack-date-match'
            if len(exact_date) > 1:
                direct = [r for r in exact_date if stem_norm in norm_text(r.stem)]
                if len(direct) == 1:
                    return direct[0], 'stack-date-plus-stem-match'
                return exact_date[0], 'stack-date-multi-first'

        # 2) Direct stem inclusion as second preference.
        direct = [r for r in rasters if stem_norm in norm_text(r.stem)]
        if len(direct) == 1:
            return direct[0], 'stack-direct-match'
        if len(direct) > 1:
            rasters = direct

        # 3) Fuzzy fallback, but only with year agreement and strong score.
        best_name, score, idx = extract_best_match(stem, [r.stem for r in rasters])
        if best_name is None:
            continue
        if idx is not None and idx < len(rasters):
            candidate = rasters[idx]
        else:
            candidate = next((r for r in rasters if r.stem == best_name), rasters[0])

        if year is not None and year not in candidate.stem:
            continue
        if score >= 90:
            return candidate, f'stack-fuzzy-match:{int(score)}'

    return None, 'stack-raster-not-found'


OFFSET_VECTOR_RE = re.compile(r'<gml:offsetVector([^>]*)>([^<]+)</gml:offsetVector>')
ALT_OFFSET_ORDER_MARKER = 'GDAL_JP2K_ALT_OFFSETVECTOR_ORDER=TRUE'
GEOGRAPHIC_SRS_RE = re.compile(r'EPSG::(4326|4167|4979|4269)|CRS84')
METRES_PER_DEGREE = 111320.0

# LINZ/LDS shorelines are digitised off the basemap, so there is no local mosaic to measure.
# These published GSDs are only a fallback for when the attribute table has no Pixel_ER.
# Auckland aerial surveys came from Z:\Chenier_Matt\Imagery\LINZ\Auckland\Raw. EDIT as new surveys are used.
LDS_DEFAULT_PIXEL_ER = 0.1
LDS_PIXEL_ER_BY_YEAR = {
    1999: 2.5,
    2000: 2.5,
    2003: 2.5,
    2012: 0.5,
    2017: 0.075,
    2020: 0.075,
    2022: 0.075,
    2024: 0.075,
}
# Only needed where a region's survey differs from the year defaults above.
LDS_PIXEL_ER_BY_REGION_YEAR = {}


def pixel_size_from_gml(text: str):
    # GMLJP2 stores the affine transform as two offset vectors; their length is the cell size.
    vectors = []
    for attrs, match in OFFSET_VECTOR_RE.findall(text):
        parts = [p for p in match.replace(',', ' ').split() if p]
        try:
            values = [float(p) for p in parts]
        except ValueError:
            continue
        if not values:
            continue
        size = math.sqrt(sum(v * v for v in values))
        # Some mosaics are stored in a geographic CRS, so the offset is in degrees.
        if GEOGRAPHIC_SRS_RE.search(attrs):
            size *= METRES_PER_DEGREE
        vectors.append(size)
        if len(vectors) == 2:
            break

    if not vectors:
        return None

    # With the alt order flag the first vector is the row (northing) step, so the second is the x cell size.
    if len(vectors) == 2 and ALT_OFFSET_ORDER_MARKER in text:
        return vectors[1]
    return vectors[0]


def pixel_size_from_sidecar(raster_path: Path):
    sidecar = Path(str(raster_path) + '.aux.xml')
    if not sidecar.exists():
        return None
    try:
        text = sidecar.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return None
    return pixel_size_from_gml(text)


def pixel_size_from_jp2_header(raster_path: Path, max_bytes: int = 1_000_000):
    if raster_path.suffix.lower() != '.jp2':
        return None
    try:
        with open(raster_path, 'rb') as f:
            head = f.read(max_bytes)
    except Exception:
        return None
    return pixel_size_from_gml(head.decode('utf-8', errors='ignore'))


def resolve_pixel_er(filename, shapefile=None, meta_row=None):
    # Pull pixel size from source mosaic metadata in Stack folders (cell size).
    raster_path, match_method = pick_stack_raster(filename)
    if raster_path is None:
        return pd.NA, match_method
    return pixel_er_from_raster(raster_path, match_method)


def pixel_er_from_raster(raster_path, match_method):
    # Reading the GML offset vectors avoids needing a JPEG2000 driver and is far quicker than opening the mosaic.
    pixel = pixel_size_from_sidecar(raster_path)
    if pixel:
        return float(abs(pixel)), f'gml-sidecar:{match_method}:{raster_path.name}'

    pixel = pixel_size_from_jp2_header(raster_path)
    if pixel:
        return float(abs(pixel)), f'gml-jp2-header:{match_method}:{raster_path.name}'

    try:
        with rasterio.open(raster_path) as src:
            xres, yres = src.res
            pixel = float(abs(xres))
            return pixel, f'rasterio:{match_method}:{raster_path.name}'
    except Exception as e:
        return pd.NA, f'raster-read-error:{e}'


def pixel_er_from_attributes(shapefile: Optional[gpd.GeoDataFrame]):
    if shapefile is None:
        return None, None
    col = next((c for c in ('Pixel_ER', 'Pixel_Er', 'pixel_er') if c in shapefile.columns), None)
    if col is None:
        return None, None
    values = pd.to_numeric(shapefile[col], errors='coerce').dropna()
    values = values[values > 0]
    if values.empty:
        return None, None
    return float(values.mode().iloc[0]), f'attribute-table:{col}'


def resolve_lds_pixel_er(filename, shapefile=None):
    # The digitiser's own Pixel_ER wins; the published GSD table is only a fallback.
    attr_pixel, attr_method = pixel_er_from_attributes(shapefile)
    if attr_pixel is not None:
        return attr_pixel, attr_method

    parts = Path(filename).parts
    region = parts[-4] if len(parts) >= 4 else ''
    shoreline_date = parse_date_from_stem(Path(filename).stem)
    year = shoreline_date.year if shoreline_date is not None else None

    if (region, year) in LDS_PIXEL_ER_BY_REGION_YEAR:
        return LDS_PIXEL_ER_BY_REGION_YEAR[(region, year)], f'lds-published-gsd:{region}:{year}'
    if year in LDS_PIXEL_ER_BY_YEAR:
        return LDS_PIXEL_ER_BY_YEAR[year], f'lds-published-gsd:{year}'
    return LDS_DEFAULT_PIXEL_ER, f'lds-published-gsd:default:{year}'


def resolve_photoscale_and_georef(filename, shapefile=None, meta_row=None, source=None):
    # Derive from source type + external metadata/CSV lookups, not shapefile attributes.
    source = source or get_source(filename, shapefile)

    if source in {'MAX', 'Max', 'max', 'PLE', 'CRI', 'NEO', 'PNE', 'GE1', 'JIN', 'JIL', 'SAT', 'VEX'}:
        return pd.NA, 1.17, 'fixed-by-source'

    if source == 'LDS':
        return pd.NA, 0, 'fixed-by-source'

    if source in {'RL', 'RLN', 'RLS', 'Rl', 'RS'}:
        try:
            dsas_date = parse_dsas_date_from_filename(filename)
            year_match = re.search(r'(\d{4})', str(filename))
            year = year_match.group(1) if year_match else None
            if dsas_date is None or year is None:
                raise ValueError('Could not parse date/year from filename for scale lookup')
            scale = get_scale(filename, dsas_date, year)
            return scale, get_georef_er(scale), 'retrolens-csv-scale'
        except Exception as e:
            print(f'Could not resolve photoscale/georef for {filename}: {e}')
            return pd.NA, pd.NA, 'scale-lookup-failed'

    return pd.NA, pd.NA, 'unknown-source'


def summarize_uncertainty_source(filename, shapefile):
    source = get_source(filename, shapefile)
    source_method = 'shapefile Source attribute' if source_from_attributes(shapefile) else 'folder path'

    # LINZ imagery is not mosaicked locally, so pixel size comes from the attribute table or the published GSD.
    if source == 'LDS':
        pixel_er, pixel_method = resolve_lds_pixel_er(filename, shapefile)
        return {
            'Source': 'LDS',
            'Pixel_ER': pixel_er,
            'Photoscale': pd.NA,
            'Georef_ER': 0,
            'Pixel_ER_method': pixel_method,
            'Georef_ER_method': f'fixed-by-source ({source_method})',
        }

    raster_path, match_method = pick_stack_raster(filename)

    # Unknown source and no mosaic to measure: treat as LINZ basemap.
    if raster_path is None and source in {'Unknown', ''}:
        pixel_er, pixel_method = resolve_lds_pixel_er(filename, shapefile)
        return {
            'Source': 'LDS',
            'Pixel_ER': pixel_er,
            'Photoscale': pd.NA,
            'Georef_ER': 0,
            'Pixel_ER_method': f'{pixel_method} (assumed LDS: {match_method})',
            'Georef_ER_method': 'fixed-by-source (assumed LDS: no Source attribute and no stack mosaic)',
        }

    pixel_er, pixel_method = pixel_er_from_raster(raster_path, match_method) if raster_path is not None else (pd.NA, match_method)
    photoscale, georef_er, georef_method = resolve_photoscale_and_georef(filename, shapefile, None, source)
    return {
        'Source': source,
        'Pixel_ER': pixel_er,
        'Photoscale': photoscale,
        'Georef_ER': georef_er,
        'Pixel_ER_method': pixel_method,
        'Georef_ER_method': f'{georef_method} ({source_method})',
    }


time: 0 ns (started: 2026-08-19 16:23:51 +12:00)


In [26]:
LDS_SURVEY_YEARS = {1999, 2000, 2003, 2012, 2017, 2020, 2022, 2024}


def attribute_date_info(shapefile):
    if shapefile is None:
        return None, None
    date_col = next((c for c in ('DSAS_Date', 'Date') if c in shapefile.columns), None)
    if date_col is None:
        return None, None
    dates = pd.to_datetime(shapefile[date_col], errors='coerce').dropna()
    if dates.empty:
        return None, None
    date = dates.mode().iloc[0]
    return date.strftime('%d/%m/%Y').lstrip('0'), str(date.year)


def attribute_date_years(shapefile):
    if shapefile is None:
        return set()
    date_col = next((c for c in ('DSAS_Date', 'Date') if c in shapefile.columns), None)
    if date_col is None:
        return set()
    dates = pd.to_datetime(shapefile[date_col], errors='coerce').dropna()
    return set(dates.dt.year.astype(int).tolist())


def source_from_attributes_or_survey(filename, shapefile=None):
    attr_source = source_from_attributes(shapefile)
    if attr_source:
        return attr_source, 'shapefile Source attribute'

    date_years = attribute_date_years(shapefile)
    if date_years and date_years.issubset(LDS_SURVEY_YEARS):
        return 'LDS', 'LDS survey year from DSAS_Date/Date attribute'

    return None, None


def get_source(filename, shapefile=None):
    attr_source, _ = source_from_attributes_or_survey(filename, shapefile)
    if attr_source:
        return attr_source

    rel = to_source_relpath(filename)
    if rel.startswith('Retrolens/') or rel.startswith('RL/') or 'Retrolens/' in rel:
        return 'RL'
    if rel.startswith('MaxarImagery/HighFreq/') or 'MaxarImagery/HighFreq/' in rel:
        return 'MAX'
    if rel.startswith('LDS/') or '/LDS/' in rel:
        return 'LDS'
    return 'Unknown'


def resolve_lds_pixel_er(filename, shapefile=None):
    attr_pixel, attr_method = pixel_er_from_attributes(shapefile)
    if attr_pixel is not None:
        return attr_pixel, attr_method

    _, attr_year = attribute_date_info(shapefile)
    if attr_year is not None:
        year = int(attr_year)
    else:
        shoreline_date = parse_date_from_stem(Path(filename).stem)
        year = shoreline_date.year if shoreline_date is not None else None

    parts = Path(filename).parts
    region = parts[-4] if len(parts) >= 4 else ''
    if (region, year) in LDS_PIXEL_ER_BY_REGION_YEAR:
        return LDS_PIXEL_ER_BY_REGION_YEAR[(region, year)], f'lds-published-gsd:{region}:{year}'
    if year in LDS_PIXEL_ER_BY_YEAR:
        return LDS_PIXEL_ER_BY_YEAR[year], f'lds-published-gsd:{year}'
    return LDS_DEFAULT_PIXEL_ER, f'lds-published-gsd:default:{year}'


def resolve_photoscale_and_georef(filename, shapefile=None, meta_row=None, source=None):
    source = source or get_source(filename, shapefile)

    if source in {'MAX', 'Max', 'max', 'PLE', 'CRI', 'NEO', 'PNE', 'GE1', 'JIN', 'JIL', 'SAT', 'VEX'}:
        return pd.NA, 1.17, 'fixed-by-source'

    if source == 'LDS':
        return pd.NA, 0, 'fixed-by-source'

    if source in {'RL', 'RLN', 'RLS', 'Rl', 'RS'}:
        try:
            attr_dsas_date, attr_year = attribute_date_info(shapefile)
            dsas_date = attr_dsas_date or parse_dsas_date_from_filename(filename)
            year = attr_year or next(iter(re.findall(r'(\d{4})', str(filename))), None)
            if dsas_date is None or year is None:
                raise ValueError('Could not parse date/year from shoreline attributes or filename')
            scale = get_scale(filename, dsas_date, year)
            return scale, get_georef_er(scale), 'retrolens-csv-scale'
        except Exception as e:
            print(f'Could not resolve photoscale/georef for {filename}: {e}')
            return pd.NA, pd.NA, 'scale-lookup-failed'

    return pd.NA, pd.NA, 'unknown-source'


time: 0 ns (started: 2026-08-19 16:23:51 +12:00)


In [27]:
def parse_date_from_stem(stem: str):
    s = str(stem)

    patterns = [
        (r'(\d{1,2}[A-Za-z]{3,4}\d{4})', ('%d%b%Y', '%d%B%Y')),
        (r'(\d{4}[A-Za-z]{3,4}\d{1,2})', ('%Y%b%d', '%Y%B%d')),
    ]
    for pattern, formats in patterns:
        match = re.search(pattern, s)
        if match:
            token = normalize_month_typos(match.group(1))
            for fmt in formats:
                try:
                    return pd.to_datetime(token, format=fmt).date()
                except Exception:
                    pass

    match = re.search(r'(\d{4}-\d{2}-\d{2})', s)
    if match:
        try:
            return pd.to_datetime(match.group(1), format='%Y-%m-%d').date()
        except Exception:
            pass

    match = re.search(r'(\d{8})', s)
    if match:
        try:
            return pd.to_datetime(match.group(1), format='%Y%m%d').date()
        except Exception:
            pass

    return None


time: 0 ns (started: 2026-08-19 16:23:51 +12:00)


In [37]:
def row_pixel_info(filename, date, source, row):
    value, column = row_value(row, ('Pixel_ER', 'Pixel_Er', 'pixel_er'))
    if value is not None:
        parsed = pd.to_numeric(value, errors='coerce')
        if pd.notna(parsed) and float(parsed) > 0:
            return float(parsed), f'attribute table:{column}', ''

    if source == 'LDS':
        if date is None:
            return pd.NA, 'LDS fallback unavailable: missing date', 'missing date'
        if date.year in LDS_PIXEL_ER_BY_YEAR:
            return LDS_PIXEL_ER_BY_YEAR[date.year], f'LDS survey GSD:{date.year}', ''
        return LDS_DEFAULT_PIXEL_ER, f'LDS default GSD:{date.year}', ''

    if source in {'RL', 'MAX'} and date is not None:
        raster_path, match_method = stack_raster_for_date(filename, date, source)
        if raster_path is not None:
            pixel, method = pixel_er_from_raster(raster_path, match_method)
            return pixel, method, str(raster_path)
        return pd.NA, match_method, 'stack mosaic not found'

    return pd.NA, 'unresolved: source/date unavailable', 'source/date unavailable'


time: 0 ns (started: 2026-08-19 16:39:51 +12:00)


In [36]:
def stack_raster_for_date(filename, shoreline_date, source=None):
    if shoreline_date is None:
        return None, 'stack-raster-not-found'

    source = str(source or '').upper()
    path = Path(filename)
    roots = []
    if len(path.parts) >= 4:
        region, aoi = path.parts[-4], path.parts[-3]
        if source == 'MAX':
            roots.append(Path(r'Z:\MaxarImagery\HighFreq') / region / aoi / 'Stack')
        elif source == 'RL':
            roots.append(Path(r'Z:\Retrolens') / region / aoi / 'Stack')
    roots.extend(infer_stack_dirs(filename))

    seen = set()
    for stack_dir in roots:
        key = str(stack_dir).lower()
        if key in seen or not stack_dir.exists():
            continue
        seen.add(key)
        rasters = []
        for ext in ('*.jp2', '*.JP2', '*.tif', '*.TIF', '*.tiff', '*.TIFF'):
            rasters.extend(stack_dir.glob(ext))

        dated_rasters = [(r, parse_date_from_stem(r.stem)) for r in rasters]
        dated_rasters = [(r, d) for r, d in dated_rasters if d is not None]
        exact = [r for r, d in dated_rasters if d == shoreline_date]
        if len(exact) == 1:
            return exact[0], f'stack-row-exact-date-match:{source or "path"}'
        if len(exact) > 1:
            return exact[0], f'stack-row-exact-date-multi-first:{source or "path"}'

        if source == 'RL':
            nearby = [
                (r, d, abs((d - shoreline_date).days))
                for r, d in dated_rasters
                if abs((d - shoreline_date).days) <= 92
            ]
            if nearby:
                nearby.sort(key=lambda item: item[2])
                nearest_distance = nearby[0][2]
                nearest = [item for item in nearby if item[2] == nearest_distance]
                if len(nearest) == 1:
                    raster, raster_date, _ = nearest[0]
                    return raster, f'stack-row-RL-within-3-months:{raster_date.isoformat()}'

    return None, f'stack-exact-date-raster-not-found:{source or "path"}'


time: 0 ns (started: 2026-08-19 16:39:49 +12:00)


In [32]:
if new_shorelines.empty:
    raise ValueError('No shoreline files matched the current selection.')


def row_value(row, names):
    for name in names:
        if name in row.index:
            value = row[name]
            if pd.notna(value) and str(value).strip() not in {'', 'nan', 'None'}:
                return value, name
    return None, None


def row_date_info(row, filename):
    value, column = row_value(row, ('DSAS_Date', 'Date'))
    if value is not None:
        parsed = pd.to_datetime(value, errors='coerce')
        if pd.notna(parsed):
            return parsed.date(), column, 'attribute table'

    parsed = parse_date_from_stem(Path(filename).stem)
    if parsed is not None:
        return parsed, 'filename', 'filename fallback'
    return None, column or '', 'unresolved'


def row_source_info(row, date):
    value, column = row_value(row, ('Source',))
    if value is not None:
        code = SOURCE_ALIASES.get(str(value).strip().upper(), str(value).strip().upper())
        return code, 'attribute table'
    if date is not None and date.year in LDS_SURVEY_YEARS:
        return 'LDS', 'LDS survey year fallback'
    return 'Unknown', 'unresolved: no Source attribute and non-LDS year'


def row_pixel_info(filename, date, source, row):
    value, column = row_value(row, ('Pixel_ER', 'Pixel_Er', 'pixel_er'))
    if value is not None:
        parsed = pd.to_numeric(value, errors='coerce')
        if pd.notna(parsed) and float(parsed) > 0:
            return float(parsed), f'attribute table:{column}', ''

    if source == 'LDS':
        if date is None:
            return pd.NA, 'LDS fallback unavailable: missing date', 'missing date'
        if date.year in LDS_PIXEL_ER_BY_YEAR:
            return LDS_PIXEL_ER_BY_YEAR[date.year], f'LDS survey GSD:{date.year}', ''
        return LDS_DEFAULT_PIXEL_ER, f'LDS default GSD:{date.year}', ''

    if source in {'RL', 'MAX'} and date is not None:
        raster_path, match_method = stack_raster_for_date(filename, date)
        if raster_path is not None:
            pixel, method = pixel_er_from_raster(raster_path, match_method)
            return pixel, method, str(raster_path)
        return pd.NA, f'no exact dated Stack mosaic:{date.isoformat()}', 'stack mosaic not found'

    return pd.NA, 'unresolved: source/date unavailable', 'source/date unavailable'


def row_georef_info(filename, date, source, row):
    value, column = row_value(row, ('Georef_ER', 'Georef_Er', 'georef_er'))
    if value is not None:
        parsed = pd.to_numeric(value, errors='coerce')
        if pd.notna(parsed) and float(parsed) >= 0:
            return float(parsed), f'attribute table:{column}', pd.NA

    if source == 'LDS':
        return 0.0, 'fixed by source:LDS', pd.NA
    if source == 'MAX':
        return 1.17, 'fixed by source:MAX', pd.NA
    if source == 'RL' and date is not None:
        dsas_date = pd.Timestamp(date).strftime('%d/%m/%Y').lstrip('0')
        year = str(date.year)
        try:
            scale = get_scale(filename, dsas_date, year)
            return get_georef_er(scale), f'Retrolens CSV photoscale:{scale}', scale
        except Exception as error:
            return pd.NA, f'Retrolens photoscale unresolved:{error}', pd.NA
    return pd.NA, 'unresolved: source/date unavailable', pd.NA


report_rows = []
missing_rows = []

for rec in tqdm(new_shorelines.to_dict('records'), total=len(new_shorelines), desc='Build uncertainty report'):
    filename = rec['shoreline_path']
    rel_filename = to_source_relpath(filename)
    try:
        shoreline = gpd.read_file(filename)
    except Exception as error:
        missing_rows.append({'filename': rel_filename, 'path': filename, 'error': f'Could not read shapefile: {error}'})
        continue
    if shoreline.empty:
        missing_rows.append({'filename': rel_filename, 'path': filename, 'error': 'Empty shapefile'})
        continue

    for row_number, row in shoreline.iterrows():
        date, date_column, date_method = row_date_info(row, filename)
        source, source_method = row_source_info(row, date)
        pixel_er, pixel_method, mosaic_path = row_pixel_info(filename, date, source, row)
        georef_er, georef_method, photoscale = row_georef_info(filename, date, source, row)

        cps_value, cps_column = row_value(row, ('CPS',))
        cps = pd.to_numeric(cps_value, errors='coerce') if cps_value is not None else 1
        cps_defaulted = cps_value is None or pd.isna(cps) or int(cps) not in CPS_error_lookup
        if cps_defaulted:
            cps = 1
        dig_er = CPS_error_lookup[int(cps)]

        total_uncy = pd.NA
        status = 'READY'
        blockers = []
        if date is None:
            blockers.append('missing DSAS date')
        if source == 'Unknown':
            blockers.append('missing Source attribute')
        if pd.isna(pixel_er):
            blockers.append('missing Pixel_ER')
        if pd.isna(georef_er):
            blockers.append('missing Georef_ER')
        if blockers:
            status = 'FLAGGED'
        else:
            total_uncy = float(np.sqrt(float(pixel_er) ** 2 + float(georef_er) ** 2 + dig_er ** 2))

        report_rows.append({
            'region': rec['region'],
            'aoi': rec['aoi'],
            'shoreline_file': Path(filename).name,
            'shoreline_path': filename,
            'row_number': int(row_number),
            'dsas_date': date.isoformat() if date is not None else pd.NA,
            'date_column': date_column,
            'date_method': date_method,
            'source': source,
            'source_method': source_method,
            'pixel_er': pixel_er,
            'pixel_er_method': pixel_method,
            'mosaic_path': mosaic_path,
            'georef_er': georef_er,
            'georef_er_method': georef_method,
            'photoscale': photoscale,
            'cps': int(cps),
            'cps_method': f'attribute table:{cps_column}' if not cps_defaulted else 'defaulted to CPS=1',
            'dig_er': dig_er,
            'total_uncy': total_uncy,
            'status': status,
            'flag_reason': '; '.join(blockers),
        })

uncy_row_report = pd.DataFrame(report_rows)
uncy_row_missing = uncy_row_report[uncy_row_report['status'] == 'FLAGGED'].copy()
uncy_row_report.to_csv(DATA_DIR / 'new_uncy_row_report.csv', index=False)
uncy_row_missing.to_csv(DATA_DIR / 'new_uncy_row_missing.csv', index=False)

print(f'Wrote report only: {len(uncy_row_report)} shoreline rows')
print(f'READY: {(uncy_row_report["status"] == "READY").sum()} | FLAGGED: {(uncy_row_report["status"] == "FLAGGED").sum()}')
display(uncy_row_report)


Build uncertainty report:   0%|          | 0/67 [00:00<?, ?it/s]

Wrote report only: 1121 shoreline rows
READY: 1101 | FLAGGED: 20


,region,aoi,shoreline_file,shoreline_path,row_number,dsas_date,date_column,date_method,source,source_method,pixel_er,pixel_er_method,mosaic_path,georef_er,georef_er_method,photoscale,cps,cps_method,dig_er,total_uncy,status,flag_reason
0,Auckland,Hobsonville,Hobsonville_shorelines.shp,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,0,2024-01-01,DSAS_Date,attribute table,LDS,attribute table,0.075,attribute table:Pixel_ER,,0.0,attribute table:Georef_ER,<NA>,1,defaulted to CPS=1,0.43,0.436492,READY,
1,Auckland,Hobsonville,Hobsonville_shorelines.shp,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,1,2024-01-01,DSAS_Date,attribute table,LDS,attribute table,0.075,attribute table:Pixel_ER,,0.0,attribute table:Georef_ER,<NA>,1,defaulted to CPS=1,0.43,0.436492,READY,
2,Auckland,Hobsonville,Hobsonville_shorelines.shp,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,2,2017-01-01,DSAS_Date,attribute table,LDS,attribute table,0.075,attribute table:Pixel_ER,,0.0,attribute table:Georef_ER,<NA>,1,defaulted to CPS=1,0.43,0.436492,READY,
3,Auckland,Hobsonville,Hobsonville_shorelines.shp,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,3,2017-01-01,DSAS_Date,attribute table,LDS,attribute table,0.075,attribute table:Pixel_ER,,0.0,attribute table:Georef_ER,<NA>,1,defaulted to CPS=1,0.43,0.436492,READY,
4,Auckland,Hobsonville,Hobsonville_shorelines.shp,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,4,2017-01-01,DSAS_Date,attribute table,LDS,attribute table,0.075,attribute table:Pixel_ER,,0.0,attribute table:Georef_ER,<NA>,1,defaulted to CPS=1,0.43,0.436492,READY,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1116,Northland,Tinopai,Tinopai_03DEC1995.shp,Z:\Retrolens\Northland\Tinopai\Shorelines\Tinopai_03DEC1995.shp,11,1995-12-03,Date,attribute table,RL,attribute table,1.52,attribute table:Pixel_Er,,2.9,attribute table:Georef_ER,<NA>,3,attribute table:CPS,0.97,3.414865,READY,
1117,Northland,Tinopai,Tinopai_03DEC1995.shp,Z:\Retrolens\Northland\Tinopai\Shorelines\Tinopai_03DEC1995.shp,12,1995-12-03,Date,attribute table,RL,attribute table,1.52,attribute table:Pixel_Er,,2.9,attribute table:Georef_ER,<NA>,3,attribute table:CPS,0.97,3.414865,READY,
1118,Northland,Tinopai,Tinopai_03DEC1995.shp,Z:\Retrolens\Northland\Tinopai\Shorelines\Tinopai_03DEC1995.shp,13,1995-12-03,Date,attribute table,RL,attribute table,1.52,attribute table:Pixel_Er,,2.9,attribute table:Georef_ER,<NA>,3,attribute table:CPS,0.97,3.414865,READY,
1119,Northland,Tinopai,Tinopai_03DEC1995.shp,Z:\Retrolens\Northland\Tinopai\Shorelines\Tinopai_03DEC1995.shp,14,1995-12-03,Date,attribute table,RL,attribute table,1.52,attribute table:Pixel_Er,,2.9,attribute table:Georef_ER,<NA>,3,attribute table:CPS,0.97,3.414865,READY,


time: 22 s (started: 2026-08-19 16:30:58 +12:00)


In [42]:
# Rebuild the report after all resolver overrides are loaded. This cell only writes CSV files.
report_rows = []

for rec in tqdm(new_shorelines.to_dict('records'), total=len(new_shorelines), desc='Build final uncertainty report'):
    filename = rec['shoreline_path']
    rel_filename = to_source_relpath(filename)
    try:
        shoreline = gpd.read_file(filename)
    except Exception as error:
        continue
    if shoreline.empty:
        continue

    for row_number, row in shoreline.iterrows():
        date, date_column, date_method = row_date_info(row, filename)
        source, source_method = row_source_info(row, date)
        pixel_er, pixel_method, mosaic_path = row_pixel_info(filename, date, source, row)
        georef_er, georef_method, photoscale = row_georef_info(filename, date, source, row)

        cps_value, cps_column = row_value(row, ('CPS',))
        cps = pd.to_numeric(cps_value, errors='coerce') if cps_value is not None else 1
        cps_defaulted = cps_value is None or pd.isna(cps) or int(cps) not in CPS_error_lookup
        if cps_defaulted:
            cps = 1
        dig_er = CPS_error_lookup[int(cps)]

        blockers = []
        if date is None:
            blockers.append('missing DSAS date')
        if source == 'Unknown':
            blockers.append('missing Source attribute')
        if pd.isna(pixel_er):
            blockers.append('missing Pixel_ER')
        if pd.isna(georef_er):
            blockers.append('missing Georef_ER')
        total_uncy = pd.NA if blockers else float(np.sqrt(float(pixel_er) ** 2 + float(georef_er) ** 2 + dig_er ** 2))

        report_rows.append({
            'region': rec['region'], 'aoi': rec['aoi'],
            'shoreline_file': Path(filename).name, 'shoreline_path': filename,
            'row_number': int(row_number),
            'dsas_date': date.isoformat() if date is not None else pd.NA,
            'date_column': date_column, 'date_method': date_method,
            'source': source, 'source_method': source_method,
            'pixel_er': pixel_er, 'pixel_er_method': pixel_method,
            'mosaic_path': mosaic_path, 'georef_er': georef_er,
            'georef_er_method': georef_method, 'photoscale': photoscale,
            'cps': int(cps),
            'cps_method': f'attribute table:{cps_column}' if not cps_defaulted else 'defaulted to CPS=1',
            'dig_er': dig_er, 'total_uncy': total_uncy,
            'status': 'READY' if not blockers else 'FLAGGED',
            'flag_reason': '; '.join(blockers),
        })

uncy_row_report = pd.DataFrame(report_rows)
uncy_row_missing = uncy_row_report[uncy_row_report['status'] == 'FLAGGED'].copy()
uncy_row_report.to_csv(DATA_DIR / 'new_uncy_row_report.csv', index=False)
uncy_row_missing.to_csv(DATA_DIR / 'new_uncy_row_missing.csv', index=False)
print(f'Wrote report only: {len(uncy_row_report)} shoreline rows')
print(f'READY: {(uncy_row_report["status"] == "READY").sum()} | FLAGGED: {(uncy_row_report["status"] == "FLAGGED").sum()}')


Build final uncertainty report:   0%|          | 0/67 [00:00<?, ?it/s]

Wrote report only: 1121 shoreline rows
READY: 1121 | FLAGGED: 0
time: 23.2 s (started: 2026-08-19 16:41:01 +12:00)


In [41]:
def stack_raster_for_source_date(filename, shoreline_date, source):
    if shoreline_date is None:
        return None, 'stack-raster-not-found'
    source = str(source or '').upper()
    path = Path(filename)
    region, aoi = path.parts[-4], path.parts[-3]
    if source == 'MAX':
        stack_dir = Path(r'Z:\MaxarImagery\HighFreq') / region / aoi / 'Stack'
    else:
        stack_dir = Path(r'Z:\Retrolens') / region / aoi / 'Stack'
    if not stack_dir.exists():
        return None, f'stack-folder-not-found:{stack_dir}'

    rasters = []
    for ext in ('*.jp2', '*.JP2', '*.tif', '*.TIF', '*.tiff', '*.TIFF'):
        rasters.extend(stack_dir.glob(ext))
    dated = [(r, parse_date_from_stem(r.stem)) for r in rasters]
    dated = [(r, d) for r, d in dated if d is not None]
    exact = [(r, d) for r, d in dated if d == shoreline_date]
    if exact:
        return exact[0][0], f'stack-exact-date:{exact[0][1].isoformat()}'
    if source == 'RL':
        nearby = [(r, d, abs((d - shoreline_date).days)) for r, d in dated if abs((d - shoreline_date).days) <= 92]
        if nearby:
            nearby.sort(key=lambda item: item[2])
            return nearby[0][0], f'stack-RL-within-3-months:{nearby[0][1].isoformat()}'
    return None, f'stack-date-not-found:{shoreline_date.isoformat()}'


def row_pixel_info(filename, date, source, row):
    value, column = row_value(row, ('Pixel_ER', 'Pixel_Er', 'pixel_er'))
    if value is not None:
        parsed = pd.to_numeric(value, errors='coerce')
        if pd.notna(parsed) and float(parsed) > 0:
            return float(parsed), f'attribute table:{column}', ''
    if source == 'LDS':
        if date is not None and date.year in LDS_PIXEL_ER_BY_YEAR:
            return LDS_PIXEL_ER_BY_YEAR[date.year], f'LDS survey GSD:{date.year}', ''
        return LDS_DEFAULT_PIXEL_ER, f'LDS default GSD:{date.year if date else "unknown"}', ''
    if source in {'RL', 'MAX'} and date is not None:
        raster_path, match_method = stack_raster_for_source_date(filename, date, source)
        if raster_path is not None:
            pixel, method = pixel_er_from_raster(raster_path, match_method)
            return pixel, method, str(raster_path)
        return pd.NA, match_method, 'stack mosaic not found'
    return pd.NA, 'unresolved: source/date unavailable', 'source/date unavailable'


time: 0 ns (started: 2026-08-19 16:40:59 +12:00)
